# Час 08 - Узорак дизајна више агената


## Подешавање


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Зашто вишеагентски системи?

Задаци из стварног света као што је планирање путовања укључују многе различите врсте експертизе — логистику, локално знање, буџетирање и још много тога. Један агент који покушава да све обради брзо постаје неуправљив.

Вишеагентски системи ово решавају кроз **специјализацију**: сваки агент се фокусира на једно поље експертизе, производећи резултате вишег квалитета од генералисте. Такође побољшавају **скалабилност** — можете додати нове агенте (нпр. специјалисту за летове, критичара ресторана) без преписивања постојећег ток рада. Агенти се састављају кроз структуирани ланац, преносећи контекст од једног до другог.


## Креирање специјализованих агената


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Прављење секвенцијалног радног тока

`WorkflowBuilder` вам омогућава да повежете агенте у усмерени граф. Овде правимо једноставну двостепену линију: **TravelPlanner** прави нацрт итинерера, затим га **TravelConcierge** прегледа и унапреди.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Додавање више агената у ток рада

Једна од највећих предности мулти-агент патерна је колико је лако проширити га. Испод додајемо агента **BudgetReviewer** који проверава план са буџетом путника, означава ставке које би могле прелазити лимит трошкова и предлаже алтернативе за уштеду. Ток рада сада покреће три агента узастопно:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Резиме

У овој лекцији сте научили како да:

1. **Креирате специјализоване агенте** — сваки са фокусираном улогом (планирање, консијерже, преглед буџета).
2. **Повежете агенте у секвенцијални ток рада** користећи `WorkflowBuilder` и `add_edge`.
3. **Стримујете излаз** из мулти-агентског процеса, пратећи који агент говори.
4. **Проширите ток рада** додавањем нових агената у ланац без измене постојећих.

Мулти-агентски дизајн образац држи сваки агент једноставним док производи богатије, темељније прегледане резултате него што би један агент могао сам постићи.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
